# Build & Opponent Analysis — Eagle Overnight Run (2026-04-11)

Weapon/hullmod popularity, opponent difficulty, and top build breakdown.

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from pathlib import Path
from collections import Counter
from optuna.trial import TrialState

EXPERIMENT_DIR = Path("../experiments/eagle-overnight-2026-04-11")
STUDY_DB = EXPERIMENT_DIR / "study.db"
EVAL_LOG = EXPERIMENT_DIR / "evaluation_log.jsonl"

study = optuna.load_study(study_name="eagle", storage=f"sqlite:///{STUDY_DB}")
trials = study.trials
completed = [t for t in trials if t.state == TrialState.COMPLETE]

eval_entries = []
with open(EVAL_LOG) as f:
    for line in f:
        eval_entries.append(json.loads(line))

print(f"Loaded {len(eval_entries)} eval entries, {len(completed)} completed trials")

## Top 10 Builds

Detailed breakdown of the best-performing builds.

In [ ]:
# Top builds from eval log (has full build details)
top_entries = sorted(eval_entries, key=lambda e: e["fitness"], reverse=True)[:10]

for i, e in enumerate(top_entries):
    build = e["build"]
    weapons = {s: w for s, w in build["weapon_assignments"].items() if w is not None}
    mods = build["hullmods"]
    wins = sum(1 for r in e["opponent_results"] if r["winner"] == "PLAYER")
    losses = sum(1 for r in e["opponent_results"] if r["winner"] == "ENEMY")
    stops = sum(1 for r in e["opponent_results"] if r["winner"] == "STOPPED")
    print(f"#{i+1} fitness={e['fitness']:.4f} (trial {e['trial_number']}) — W{wins}/L{losses}/S{stops}")
    print(f"  Weapons: {', '.join(f'{s}={w}' for s, w in sorted(weapons.items()))}")
    print(f"  Hullmods: {', '.join(sorted(mods))}")
    print(f"  Vents={build['flux_vents']}, Caps={build['flux_capacitors']}")
    print()

## Weapon Popularity

Which weapons appear most in top builds vs all builds?

In [ ]:
# Weapon frequency in all builds vs top 25%
all_weapons = Counter()
top_weapons = Counter()

top_25_threshold = np.percentile([e["fitness"] for e in eval_entries], 75)

for e in eval_entries:
    weapons = [w for w in e["build"]["weapon_assignments"].values() if w is not None]
    all_weapons.update(weapons)
    if e["fitness"] >= top_25_threshold:
        top_weapons.update(weapons)

# Compare top-20 weapons
top_20_all = all_weapons.most_common(20)
weapon_names = [w for w, _ in top_20_all]
all_counts = [all_weapons[w] for w in weapon_names]
top_counts = [top_weapons.get(w, 0) for w in weapon_names]

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(weapon_names))
width = 0.35
ax.barh([i - width/2 for i in x], all_counts, width, label="All builds", alpha=0.7)
ax.barh([i + width/2 for i in x], top_counts, width, label=f"Top 25% (fitness >= {top_25_threshold:.3f})", alpha=0.7)
ax.set_yticks(list(x))
ax.set_yticklabels(weapon_names)
ax.set_xlabel("Count")
ax.set_title("Weapon Frequency: All Builds vs Top 25%")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Enrichment: weapons over/under-represented in top builds
print(f"\nTop 25% threshold: fitness >= {top_25_threshold:.3f}")
print(f"Top 25% builds: {sum(1 for e in eval_entries if e['fitness'] >= top_25_threshold)}")
print("\nWeapon enrichment in top 25%:")
total_all = sum(all_weapons.values())
total_top = sum(top_weapons.values())
for w in weapon_names[:15]:
    frac_all = all_weapons[w] / total_all
    frac_top = top_weapons.get(w, 0) / total_top if total_top > 0 else 0
    enrichment = frac_top / frac_all if frac_all > 0 else 0
    arrow = "+" if enrichment > 1.2 else ("-" if enrichment < 0.8 else "=")
    print(f"  {arrow} {w}: {frac_all:.1%} (all) -> {frac_top:.1%} (top) [{enrichment:.2f}x]")

## Hullmod Popularity

In [ ]:
all_mods = Counter()
top_mods = Counter()

for e in eval_entries:
    mods = e["build"]["hullmods"]
    all_mods.update(mods)
    if e["fitness"] >= top_25_threshold:
        top_mods.update(mods)

# Compare
top_20_mods = all_mods.most_common(20)
mod_names = [m for m, _ in top_20_mods]
all_mod_counts = [all_mods[m] for m in mod_names]
top_mod_counts = [top_mods.get(m, 0) for m in mod_names]

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(mod_names))
width = 0.35
ax.barh([i - width/2 for i in x], all_mod_counts, width, label="All builds", alpha=0.7)
ax.barh([i + width/2 for i in x], top_mod_counts, width, label=f"Top 25%", alpha=0.7)
ax.set_yticks(list(x))
ax.set_yticklabels(mod_names)
ax.set_xlabel("Count")
ax.set_title("Hullmod Frequency: All Builds vs Top 25%")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Enrichment
n_all = len(eval_entries)
n_top = sum(1 for e in eval_entries if e["fitness"] >= top_25_threshold)
print(f"\nHullmod enrichment in top 25% ({n_top} builds):")
for m in mod_names:
    frac_all = all_mods[m] / n_all
    frac_top = top_mods.get(m, 0) / n_top if n_top > 0 else 0
    enrichment = frac_top / frac_all if frac_all > 0 else 0
    arrow = "+" if enrichment > 1.2 else ("-" if enrichment < 0.8 else "=")
    print(f"  {arrow} {m}: {frac_all:.0%} (all) -> {frac_top:.0%} (top) [{enrichment:.2f}x]")

## Opponent Difficulty

Win rate and mean HP differential against each opponent.

In [ ]:
# Aggregate per-opponent stats
opponent_stats = {}

for e in eval_entries:
    for r in e["opponent_results"]:
        opp = r["opponent"]
        if opp not in opponent_stats:
            opponent_stats[opp] = {"wins": 0, "losses": 0, "stops": 0, "durations": [], "hp_diffs": []}
        s = opponent_stats[opp]
        if r["winner"] == "PLAYER":
            s["wins"] += 1
        elif r["winner"] == "ENEMY":
            s["losses"] += 1
        else:
            s["stops"] += 1
        s["durations"].append(r["duration_seconds"])
        s["hp_diffs"].append(r["hp_differential"])

# Build dataframe
rows = []
for opp, s in opponent_stats.items():
    total = s["wins"] + s["losses"] + s["stops"]
    rows.append({
        "opponent": opp,
        "total": total,
        "win_rate": s["wins"] / total,
        "loss_rate": s["losses"] / total,
        "stop_rate": s["stops"] / total,
        "mean_hp_diff": np.mean(s["hp_diffs"]),
        "mean_duration": np.mean(s["durations"]),
    })

opp_df = pd.DataFrame(rows).sort_values("win_rate", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Win rate by opponent
colors = plt.cm.RdYlGn(opp_df["win_rate"])
axes[0].barh(opp_df["opponent"], opp_df["win_rate"], color=colors, edgecolor="black", linewidth=0.5)
axes[0].axvline(0.5, color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Player Win Rate")
axes[0].set_title("Win Rate by Opponent (all builds)")
axes[0].set_xlim(0, 1)

# Mean HP differential by opponent
colors2 = plt.cm.RdYlGn((opp_df["mean_hp_diff"] + 1) / 2)  # normalize -1..1 to 0..1
axes[1].barh(opp_df["opponent"], opp_df["mean_hp_diff"], color=colors2, edgecolor="black", linewidth=0.5)
axes[1].axvline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Mean HP Differential")
axes[1].set_title("Mean HP Differential by Opponent")

plt.tight_layout()
plt.show()

print(opp_df[["opponent", "total", "win_rate", "mean_hp_diff", "mean_duration"]].to_string(index=False, float_format="%.3f"))

## Flux Allocation

Vent/capacitor distribution in top vs bottom builds.

In [ ]:
vents = [e["build"]["flux_vents"] for e in eval_entries]
caps = [e["build"]["flux_capacitors"] for e in eval_entries]
fitness = [e["fitness"] for e in eval_entries]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scatter: vents vs caps colored by fitness
sc = axes[0].scatter(vents, caps, c=fitness, cmap="RdYlGn", alpha=0.6, s=20, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Fitness")
axes[0].set_xlabel("Flux Vents")
axes[0].set_ylabel("Flux Capacitors")
axes[0].set_title("Flux Allocation vs Fitness")

# Vents distribution: top vs bottom
top_mask = np.array(fitness) >= top_25_threshold
bot_mask = np.array(fitness) < np.percentile(fitness, 25)
axes[1].hist([np.array(vents)[top_mask], np.array(vents)[bot_mask]],
             bins=range(0, 35), label=["Top 25%", "Bottom 25%"], alpha=0.7)
axes[1].set_xlabel("Flux Vents")
axes[1].set_ylabel("Count")
axes[1].set_title("Vent Distribution")
axes[1].legend()

# Caps distribution
axes[2].hist([np.array(caps)[top_mask], np.array(caps)[bot_mask]],
             bins=range(0, 35), label=["Top 25%", "Bottom 25%"], alpha=0.7)
axes[2].set_xlabel("Flux Capacitors")
axes[2].set_ylabel("Count")
axes[2].set_title("Capacitor Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Top 25% — Vents: {np.mean(np.array(vents)[top_mask]):.1f} mean, Caps: {np.mean(np.array(caps)[top_mask]):.1f} mean")
print(f"Bottom 25% — Vents: {np.mean(np.array(vents)[bot_mask]):.1f} mean, Caps: {np.mean(np.array(caps)[bot_mask]):.1f} mean")

## Fitness Discrimination Issue

Many builds achieved fitness=1.0 — are they meaningfully different? What weapons/hullmods do the "perfect" builds share?

In [ ]:
# Perfect builds (fitness >= 0.999)
perfect = [e for e in eval_entries if e["fitness"] >= 0.999]
near_perfect = [e for e in eval_entries if 0.7 <= e["fitness"] < 0.999]
mediocre = [e for e in eval_entries if e["fitness"] < 0.3]

print(f"Perfect (>=0.999): {len(perfect)}")
print(f"Near-perfect (0.7-0.999): {len(near_perfect)}")
print(f"Mediocre (<0.3): {len(mediocre)}")

# What do perfect builds have in common?
if perfect:
    perfect_weapons = Counter()
    perfect_mods = Counter()
    for e in perfect:
        ws = [w for w in e["build"]["weapon_assignments"].values() if w is not None]
        perfect_weapons.update(ws)
        perfect_mods.update(e["build"]["hullmods"])

    print(f"\nWeapons in ALL {len(perfect)} perfect builds:")
    for w, c in perfect_weapons.most_common(10):
        print(f"  {w}: {c} ({100*c/len(perfect):.0f}% of perfect builds)")

    print(f"\nHullmods in perfect builds:")
    for m, c in perfect_mods.most_common(10):
        print(f"  {m}: {c} ({100*c/len(perfect):.0f}% of perfect builds)")

    # Check for "garbage" items in perfect builds
    garbage_weapons = {"mininglaser", "miningblaster"}
    garbage_mods = {"surveying_equipment", "additional_berthing", "augmentedengines", "solar_shielding"}
    n_garbage = sum(1 for e in perfect
                    if any(w in garbage_weapons for w in e["build"]["weapon_assignments"].values() if w)
                    or any(m in garbage_mods for m in e["build"]["hullmods"]))
    print(f"\nPerfect builds with non-combat items: {n_garbage}/{len(perfect)} ({100*n_garbage/len(perfect):.0f}%)")

## Parameter Importance (fANOVA)

Which parameters matter most for fitness?

In [ ]:
from starsector_optimizer.importance import analyze_importance, print_importance_report

result = analyze_importance(study)
print(print_importance_report(result))

# Plot top-20 importances
top_params = list(result.importances.items())[:20]
names = [p for p, _ in top_params]
values = [v for _, v in top_params]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(names)), values, edgecolor="black", alpha=0.7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel("fANOVA Importance")
ax.set_title("Top 20 Parameter Importances")
ax.invert_yaxis()
plt.tight_layout()
plt.show()